# 4.6 · 弹性网 / Elastic Net (L1 + L2)

> **课程定位 / Where this fits**
> 第 6 课，**Part 4 · 监督学习：回归**（正则化三部曲收官）。
> Lesson 6, **Part 4 · Supervised Regression** (the regularization trilogy finale).
>
> Ridge(4.4) 稳定但不稀疏，Lasso(4.5) 稀疏但在共线特征里"任性"（随机只留一个）。**弹性网(Elastic Net)** 把 L1 和 L2 **混在一起**，兼得 Lasso 的稀疏和 Ridge 的稳定——尤其擅长**高维 + 特征相关**的场景（基因、文本等）。
> Ridge (4.4) is stable but not sparse; Lasso (4.5) is sparse but "capricious" among collinear features (keeps one arbitrarily). **Elastic Net** **mixes** L1 and L2, getting Lasso's sparsity and Ridge's stability — especially for **high-dimensional, correlated** features (genomics, text).
>
> 💼 **实战/面试视角**："Elastic Net 解决 Lasso 什么问题 / l1_ratio 是什么" 在正则化追问里常出现。
> 💼 **Practical/interview angle:** "what does Elastic Net fix in Lasso / what's l1_ratio" come up in regularization follow-ups.

> 📐 **符号约定 / Notation**
> - $\lambda$ (`alpha`) —— 总正则强度 / overall strength
> - $r$ (`l1_ratio`) —— L1 占比，$r=1$ 纯 Lasso，$r=0$ 纯 Ridge

> 💡 **面试相关 / Interview-relevant**
> - "Elastic Net = L1 + L2, 解决 Lasso 什么问题"（出镜率 ★★★★）
> - "什么是分组效应(grouping effect)"（★★★★）
> - "l1_ratio 怎么调"（★★★）
> - "三种正则什么时候用哪个"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解弹性网 = L1 惩罚 + L2 惩罚的加权混合。
   Understand Elastic Net = a weighted mix of L1 and L2 penalties.
2. 看 `l1_ratio` 如何在 Ridge 和 Lasso 之间连续切换。
   See how `l1_ratio` slides continuously between Ridge and Lasso.
3. 理解**分组效应**——它怎么修好 Lasso 的共线"任性"。
   Understand the **grouping effect** — how it fixes Lasso's collinear capriciousness.
4. 用 `ElasticNetCV` 同时调 alpha 和 l1_ratio。
   Tune alpha and l1_ratio together with `ElasticNetCV`.
5. 横向总结 OLS / Ridge / Lasso / ElasticNet 的选择。
   Summarize the choice among OLS / Ridge / Lasso / Elastic Net.

## 目录 / TOC
1. [先建直觉 + 公式 ⭐](#1)
2. [l1_ratio：在 Ridge 和 Lasso 间滑动 ⭐](#2)
3. [分组效应：修好 Lasso 的任性 ⭐](#3)
4. [ElasticNetCV 调参 ⭐](#4)
5. [四模型横向对比 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + 公式 ⭐ / Intuition & Formula

弹性网的损失就是把 Lasso 的 L1 和 Ridge 的 L2 **两个惩罚都加上**：
Elastic Net's loss simply **adds both penalties** — Lasso's L1 and Ridge's L2:

$$J(\mathbf{w}) = \frac1n\|\mathbf{Xw}-\mathbf{y}\|^2 + \lambda\Big(\underbrace{r\sum_j|w_j|}_{\text{L1: 稀疏}} + \underbrace{(1-r)\sum_j w_j^2}_{\text{L2: 稳定}}\Big)$$

- **`alpha`($\lambda$)** 控制总正则强度。
  `alpha` controls overall regularization strength.
- **`l1_ratio`($r$)** 控制 L1 和 L2 的比例：$r=1$ 是纯 Lasso，$r=0$ 是纯 Ridge，中间是混合。
  `l1_ratio` controls the L1/L2 mix: $r=1$ pure Lasso, $r=0$ pure Ridge, in-between mixed.

直觉：**L1 部分负责把无用特征清零（稀疏、可解释），L2 部分负责在相关特征间稳定地分配系数（不任性）。** 两者结合，取长补短。
Intuition: **the L1 part zeros out useless features (sparse, interpretable); the L2 part stably distributes coefficients among correlated features (no caprice).** Together they complement each other.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

data = fetch_california_housing(as_frame=True)
X, y = data.data.values, data.target.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)               # 同 Ridge/Lasso, 必须标准化
Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)
print(f"{X.shape}, 已标准化")


<a id="2"></a>
## 2. l1_ratio：在 Ridge 和 Lasso 间滑动 ⭐ / l1_ratio Slides Between Ridge and Lasso

固定 `alpha`，把 `l1_ratio` 从 0 调到 1，看非零系数的数量怎么变：越靠近 1（越像 Lasso），归零的系数越多（越稀疏）；越靠近 0（越像 Ridge），系数都保留。
With `alpha` fixed, sweep `l1_ratio` from 0 to 1 and watch the number of nonzero coefficients: closer to 1 (more Lasso-like) → more zeros (sparser); closer to 0 (more Ridge-like) → all kept.


In [ ]:
from sklearn.linear_model import ElasticNet

print(f"{'l1_ratio':<10} {'非零系数 nonzero':<16} 说明")
for r in [0.0, 0.2, 0.5, 0.8, 1.0]:
    # l1_ratio 必须 >0, 用极小值近似纯 Ridge / ~Ridge at 0
    en = ElasticNet(alpha=0.1, l1_ratio=max(r, 0.001), max_iter=5000).fit(Xtr, y_tr)
    nz = (np.abs(en.coef_) > 1e-6).sum()
    tag = "≈Ridge(全保留)" if r==0 else ("纯Lasso(最稀疏)" if r==1 else "混合 mixed")
    print(f"{r:<10} {nz}/{X.shape[1]:<15} {tag}")
print("\nl1_ratio 从 0→1: 越来越像 Lasso, 归零的系数越多")


<a id="3"></a>
## 3. 分组效应：修好 Lasso 的任性 ⭐ / The Grouping Effect

**分组效应(grouping effect)** 是弹性网的招牌优势。当一组特征**高度相关**时：
The **grouping effect** is Elastic Net's signature advantage. When a group of features is **highly correlated**:
- **Lasso 会任性地只留一个**、其余归零，而且留哪个对数据的微小变化很敏感（不稳定，不可复现）。
  **Lasso arbitrarily keeps just one** and zeros the rest, and which one is sensitive to tiny data changes (unstable, irreproducible).
- **弹性网把系数平摊给整组相关特征**（这是 L2 部分的功劳），稳定得多。

下面造一组 x1≈x2≈x3（强共线，都等于真实信号 base），外加一个无关特征 x4，对比两者。
Below we make x1≈x2≈x3 (strongly collinear, all equal to the true signal `base`) plus an irrelevant x4, and compare.


In [ ]:
from sklearn.linear_model import Lasso
n = 300
base = rng.normal(0, 1, n)
x1 = base + rng.normal(0, 0.05, n); x2 = base + rng.normal(0, 0.05, n); x3 = base + rng.normal(0, 0.05, n)
x4 = rng.normal(0, 1, n)                  # 与目标无关的独立特征 / irrelevant feature
y_syn = 3*base + rng.normal(0, 0.5, n)    # 真实依赖 base(=x1≈x2≈x3)
Xc = np.c_[x1, x2, x3, x4]
print("真实: y=3·base, 而 x1≈x2≈x3≈base(共线组), x4 无关\n")

lasso = Lasso(alpha=0.1).fit(Xc, y_syn)
en = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(Xc, y_syn)
print(f"{'':12} {'x1':>7} {'x2':>7} {'x3':>7} {'x4':>7}")
print(f"{'Lasso':<12} " + " ".join(f"{c:>7.2f}" for c in lasso.coef_))
print(f"{'ElasticNet':<12} " + " ".join(f"{c:>7.2f}" for c in en.coef_))
print("\nLasso: 共线组里只留 1-2 个, 其余归0(任性, 换数据可能换个留)")
print("ElasticNet: 把系数平摊给 x1,x2,x3(分组效应, 稳定); x4 都正确归0")


<a id="4"></a>
## 4. ElasticNetCV 调参 ⭐ / Tuning with ElasticNetCV

弹性网有**两个**超参（alpha 和 l1_ratio），`ElasticNetCV` 用交叉验证**同时**搜索两者——相当于让数据自己决定"要多少稀疏、要多少稳定"。
Elastic Net has **two** hyperparameters (alpha and l1_ratio); `ElasticNetCV` searches **both** by cross-validation — letting the data decide "how much sparsity vs how much stability".


In [ ]:
from sklearn.linear_model import ElasticNetCV

# 给一组候选 l1_ratio, alpha 自动沿路径搜索 / search l1_ratio grid × alpha path
encv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
                    cv=5, max_iter=10000, random_state=0).fit(Xtr, y_tr)
print(f"ElasticNetCV 选出: alpha={encv.alpha_:.4f}, l1_ratio={encv.l1_ratio_}")
print(f"test R² = {encv.score(Xte, y_te):.4f}")
print(f"非零系数: {(np.abs(encv.coef_)>1e-6).sum()}/{X.shape[1]}")
print("l1_ratio 也由 CV 选出 → 数据自己决定要多少稀疏 vs 多少稳定")


<a id="5"></a>
## 5. 四模型横向对比 + 小结 ⭐ / Four Models Compared & Summary

在"8 真实 + 30 噪声"特征上对比 OLS / Ridge / Lasso / ElasticNet：噪声多时正则模型明显胜出，且 Lasso/ElasticNet 还自动剔除噪声特征。
On "8 real + 30 noise" features, compare OLS / Ridge / Lasso / ElasticNet: regularized models win clearly under noise, and Lasso/ElasticNet additionally drop noise features.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

noise = rng.normal(size=(len(X), 30))     # 加 30 个噪声特征放大正则的差异
Xa = np.c_[X, noise]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, y, test_size=0.3, random_state=0)
sc = StandardScaler().fit(Xa_tr); Xa_tr, Xa_te = sc.transform(Xa_tr), sc.transform(Xa_te)

models = {"OLS": LinearRegression(), "Ridge": Ridge(alpha=1.0),
          "Lasso": Lasso(alpha=0.05, max_iter=10000),
          "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000)}
print("8 真实 + 30 噪声特征:")
print(f"{'模型 model':<12} {'test R²':>9} {'非零系数':>11}")
for name, m in models.items():
    m.fit(Xa_tr, ya_tr)
    nz = f"{(np.abs(m.coef_) > 1e-6).sum()}/38"
    print(f"{name:<12} {m.score(Xa_te, ya_te):>9.4f} {nz:>11}")
print("\nLasso/ElasticNet 剔除大量噪声 → 更简洁; 噪声越多正则优势越明显")


```
弹性网 = OLS + λ[ r·Σ|wⱼ|(L1) + (1-r)·Σwⱼ²(L2) ]; alpha=总强度, l1_ratio=L1占比
l1_ratio: 1=纯Lasso(最稀疏), 0=纯Ridge(全保留), 中间混合
分组效应: 共线特征组里, 弹性网把系数平摊(稳定), 而 Lasso 任性只留一个
两超参用 ElasticNetCV 同时调; 必须标准化
四选一: 特征都有用→Ridge; 想稀疏可解释→Lasso; 高维+相关特征→ElasticNet; 无正则→OLS(易过拟合)
```

### 💡 面试速查 / Interview cheat-sheet
1. **弹性网 = L1 + L2 混合**，l1_ratio 控制比例。
   Elastic Net = mix of L1 + L2, ratio set by l1_ratio.
2. **分组效应**: 把相关特征的系数平摊, 修好 Lasso 的"任性只留一个"。
   Grouping effect: spreads coefficients over correlated features, fixing Lasso's "keep one arbitrarily".
3. **高维 + 相关特征首选弹性网**(基因/文本)。
   Prefer Elastic Net for high-dim correlated features (genomics/text).
4. **两超参(alpha, l1_ratio) 用 ElasticNetCV 同时调**。
   Tune both (alpha, l1_ratio) with ElasticNetCV.
5. **三正则记忆**: Ridge 稳定不稀疏 / Lasso 稀疏不稳定 / ElasticNet 两者折中。
   Ridge stable-not-sparse / Lasso sparse-not-stable / Elastic Net the compromise.

### 下一节 / Next
**4.7 广义线性模型(GLM)**——前面都假设目标是连续、误差正态。GLM 把线性回归推广到计数(泊松)、比例(逻辑)等非正态目标, 统一了一大类模型。
**4.7 GLM** — so far the target was continuous with normal errors. GLM extends linear regression to counts (Poisson), proportions (logistic), and other non-normal targets, unifying a broad family.
